In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms
from PIL import Image

import cv2

import time

In [2]:
import torch

print(torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU")

2.11.0+cu128
CUDA: True
NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
#Configuration

DATA_DIR = "./data"
IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.002
VAL_SPLIT = 0.2
MODEL_PATH = "./model.pth"

In [4]:
device = torch.device("cuda")
print(device)

cuda


In [5]:
class CNN(nn.Module):
  def __init__(self, num_classes):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(4096,128),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(128,num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

In [ ]:
import os

DATA_DIR = "./emotion-detector-data/processed_data"

print(os.getcwd())
print(os.listdir(DATA_DIR))

Current folder: c:\Emotion-Recognition
Found: []


IndexError: list index out of range

In [ ]:
full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
len(full_dataset)

49779

In [ ]:
class_names = full_dataset.classes
num_class = len(class_names)
class_names

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

In [ ]:
val_count = int(len(full_dataset)*VAL_SPLIT)
train_count = len(full_dataset) - val_count

train, val = random_split(full_dataset, [train_count, val_count])

In [ ]:
train_count, val_count

(39824, 9955)

In [ ]:
val.dataset.transform = val_transform

In [ ]:
train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, pin_memory=True)
val_loader = DataLoader(val, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
model = CNN(num_classes=num_class).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
def train():
  best_acc = 0.0

  for epoch in range(1, EPOCHS + 1):
    model.train()
    avg_loss = 0.0
    avg_correct = 0
    total = 0
    t0 = time.time()
    for images,labels in train_loader:
      images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      avg_loss += loss.item()*images.size(0)
      preds = outputs.argmax(dim=1)
      avg_correct += (preds == labels).sum().item()
      total += images.size(0)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
      for images,labels in val_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item()*images.size(0)
        preds = outputs.argmax(dim=1)
        val_correct += (preds == labels).sum().item()
        val_total += images.size(0)

    avg_loss /= total
    avg_acc = avg_correct/total
    val_loss /= val_total
    val_acc = val_correct/val_total
    t1 = time.time()
    elapsed = t1-t0
    print(f"EPOCH {epoch}/{EPOCHS}  train_loss={avg_loss:.3f} train_acc={avg_acc:.3f}  val_loss={val_loss:.3f} val_acc={val_acc:.3f}  time={elapsed:.1f}s")

    if val_acc>best_acc:
      best_acc = val_acc
      torch.save({
          "model_state" : model.state_dict(),
          "class_names" : class_names,
          "img_size" : IMG_SIZE
      }, MODEL_PATH)

      print(f"Saving the model with the best validation accuracy : {best_acc:.3f} to {MODEL_PATH}")

  print("Training completed")

In [ ]:
def inference():
  checkpoint = torch.load(MODEL_PATH,map_location=device)
  model.load_state_dict(checkpoint["model_state"])
  model.to(device)
  model.eval()
  class_names = checkpoint["class_names"]

  print("Webcam is starting... press q to quit")
  webcam = cv2.VideoCapture(0)
  if not webcam.isOpened():
    print("Webcam error")
    return

  while True:
    ret, frame = webcam.read()

    if not ret:
      break

    rbg = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resize_img = Image.fromarray(rbg).resize((IMG_SIZE, IMG_SIZE))
    tensor = transforms.functional.to_tensor(resize_img)
    tensor = transforms.functional.normalize(tensor,[0.485,0.456,0.406],[0.229,0.224,0.225])
    tensor = tensor.unsqueeze(0).to(device)

    with torch.no_grad():
      out = model(tensor)
      probs = torch.nn.functional.softmax(out, dim=1)
      top_prob, prob_idx = torch.max(probs, dim=1)
      label = class_names[prob_idx.item()]
      conf = top_prob.item()

    cv2.putText(frame, f"{label} {conf:.2f}", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)
    cv2.imshow("Emotion Detector (q to quit)", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
      break

  webcam.release()
  cv2.destroyAllWindows()

In [ ]:
train()

EPOCH 1/50  train_loss=1.485 train_acc=0.427  val_loss=1.235 val_acc=0.527  time=39.2s
Saving the model with the best validation accuracy : 0.527 to ./model.pth
EPOCH 2/50  train_loss=1.222 train_acc=0.533  val_loss=1.131 val_acc=0.566  time=37.6s
Saving the model with the best validation accuracy : 0.566 to ./model.pth
EPOCH 3/50  train_loss=1.135 train_acc=0.569  val_loss=1.064 val_acc=0.595  time=37.8s
Saving the model with the best validation accuracy : 0.595 to ./model.pth
EPOCH 4/50  train_loss=1.077 train_acc=0.591  val_loss=1.045 val_acc=0.597  time=37.6s
Saving the model with the best validation accuracy : 0.597 to ./model.pth
EPOCH 5/50  train_loss=1.033 train_acc=0.607  val_loss=0.997 val_acc=0.626  time=37.1s
Saving the model with the best validation accuracy : 0.626 to ./model.pth
EPOCH 6/50  train_loss=1.004 train_acc=0.621  val_loss=1.000 val_acc=0.622  time=36.2s
EPOCH 7/50  train_loss=0.972 train_acc=0.629  val_loss=0.996 val_acc=0.627  time=37.8s
Saving the model with